In [6]:
# ── Cellule 1 : Imports + chargement ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os

PATH_IN  = "../data/processed/"
PATH_OUT = "../data/processed/"

# Charger les fichiers nettoyés
normal2023  = pd.read_csv(PATH_IN + "normal2023_clean.csv",  encoding="latin-1")
normal2024  = pd.read_csv(PATH_IN + "normal2024_clean.csv",  encoding="latin-1")
normal2025  = pd.read_csv(PATH_IN + "normal2025_clean.csv",  encoding="latin-1")
normal2026  = pd.read_csv(PATH_IN + "normal2026_clean.csv",  encoding="latin-1")
express2024 = pd.read_csv(PATH_IN + "express2024_clean.csv", encoding="latin-1")
express2025 = pd.read_csv(PATH_IN + "express2025_clean.csv", encoding="latin-1")
express2026 = pd.read_csv(PATH_IN + "express2026_clean.csv", encoding="latin-1")

# Reconvertir les dates
for df in [normal2023, normal2024, normal2025, normal2026,
           express2024, express2025, express2026]:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Fusionner en un seul DataFrame
df_all = pd.concat([
    normal2023, normal2024, normal2025, normal2026,
    express2024, express2025, express2026,
], ignore_index=True)

print(f" Dataset complet : {len(df_all):,} lignes × {df_all.shape[1]} colonnes")
print(f"\nRépartition :")
print(df_all["type_source"].value_counts())
print(f"\nAnnées :")
print(df_all["Date"].dt.year.value_counts().sort_index())

C:\Users\eyato\AppData\Local\Temp\ipykernel_28348\3380824280.py:10: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  normal2023  = pd.read_csv(PATH_IN + "normal2023_clean.csv",  encoding="latin-1")
C:\Users\eyato\AppData\Local\Temp\ipykernel_28348\3380824280.py:11: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  normal2024  = pd.read_csv(PATH_IN + "normal2024_clean.csv",  encoding="latin-1")


 Dataset complet : 1,247,660 lignes × 25 colonnes

Répartition :
type_source
express    865453
normal     382207
Name: count, dtype: int64

Années :
Date
2023     50179
2024    360803
2025    485074
2026    351604
Name: count, dtype: int64


In [7]:
print(df_all.shape)

(1247660, 25)


In [8]:
df_all[['IdBureau','cp','cite','ville','est_agence']].head(30)

,IdBureau,cp,cite,ville,est_agence
0,2260,2260,RUE SOUK,DEGACHE,False
1,3023,3023,SFAX,SFAX,False
2,2001,2001,ENNASR,ARIANA,False
3,2001,2001,ENNASR,ARIANA,False
4,2001,2001,ENNASR,ARIANA,False
5,2001,2001,ENNASR,ARIANA,False
6,2001,2001,ENNASR,ARIANA,False
7,2001,2001,ENNASR,ARIANA,False
8,2001,2001,ENNASR,ARIANA,False
9,2001,2001,ENNASR,ARIANA,False


In [9]:
df_all[['IdBureau','cp','cite','ville']].drop_duplicates().shape

(85924, 4)

In [10]:
df_all[['IdBureau','cp','cite','ville']].drop_duplicates().head(50)

,IdBureau,cp,cite,ville
0,2260,2260,RUE SOUK,DEGACHE
1,3023,3023,SFAX,SFAX
2,2001,2001,ENNASR,ARIANA
34,1250,2011,2011,2011
72,2214,2214,HAMMA,TOZEUR
73,2001,2001,ARIANA,ENNASER
77,8044,8044,MIDA,MIDA
78,6013,6020,HERMMA,GABE
79,4160,4171,BENGURDEN,BENGURDEN
80,4171,4171,BENGURDEN,BENGURDEN


In [11]:
# ── Cellule 2 : Construire dim_temps ─────────────────────────────────────────

# Calendrier islamique complet
CALENDRIER_ISLAMIQUE_MOIS = [
    ("2023-01-14","2023-02-11","Rajab",7,1444),
    ("2023-02-12","2023-03-13","Chaabane",8,1444),
    ("2023-03-14","2023-04-20","Ramadan",9,1444),
    ("2023-04-21","2023-05-19","Chawwal",10,1444),
    ("2023-05-20","2023-06-17","Dhou al-Qi'da",11,1444),
    ("2023-06-18","2023-07-17","Dhou al-Hijja",12,1444),
    ("2023-07-18","2023-08-15","Mouharram",1,1445),
    ("2023-08-16","2023-09-14","Safar",2,1445),
    ("2023-09-15","2023-10-14","Rabi al-Awwal",3,1445),
    ("2023-10-15","2023-11-12","Rabi al-Thani",4,1445),
    ("2023-11-13","2023-12-12","Joumada al-Awwal",5,1445),
    ("2023-12-13","2024-01-10","Joumada al-Thani",6,1445),
    ("2024-01-11","2024-02-09","Rajab",7,1445),
    ("2024-02-10","2024-03-10","Chaabane",8,1445),
    ("2024-03-11","2024-04-09","Ramadan",9,1445),
    ("2024-04-10","2024-05-07","Chawwal",10,1445),
    ("2024-05-08","2024-06-06","Dhou al-Qi'da",11,1445),
    ("2024-06-07","2024-07-05","Dhou al-Hijja",12,1445),
    ("2024-07-06","2024-08-04","Mouharram",1,1446),
    ("2024-08-05","2024-09-02","Safar",2,1446),
    ("2024-09-03","2024-10-02","Rabi al-Awwal",3,1446),
    ("2024-10-03","2024-11-01","Rabi al-Thani",4,1446),
    ("2024-11-02","2024-11-30","Joumada al-Awwal",5,1446),
    ("2024-12-01","2024-12-30","Joumada al-Thani",6,1446),
    ("2024-12-31","2025-01-29","Rajab",7,1446),
    ("2025-01-30","2025-02-27","Chaabane",8,1446),
    ("2025-03-01","2025-03-29","Ramadan",9,1446),
    ("2025-03-30","2025-04-27","Chawwal",10,1446),
    ("2025-04-28","2025-05-26","Dhou al-Qi'da",11,1446),
    ("2025-05-27","2025-06-25","Dhou al-Hijja",12,1446),
    ("2025-06-26","2025-07-24","Mouharram",1,1447),
    ("2025-07-25","2025-08-22","Safar",2,1447),
    ("2025-08-23","2025-09-21","Rabi al-Awwal",3,1447),
    ("2025-09-22","2025-10-20","Rabi al-Thani",4,1447),
    ("2025-10-21","2025-11-19","Joumada al-Awwal",5,1447),
    ("2025-11-20","2025-12-19","Joumada al-Thani",6,1447),
    ("2025-12-20","2026-01-17","Rajab",7,1447),
    ("2026-01-18","2026-02-16","Chaabane",8,1447),
    ("2026-02-17","2026-03-17","Ramadan",9,1447),
    ("2026-03-18","2026-04-16","Chawwal",10,1447),
    ("2026-04-17","2026-05-15","Dhou al-Qi'da",11,1447),
    ("2026-05-16","2026-06-14","Dhou al-Hijja",12,1447),
    ("2026-06-15","2026-07-13","Mouharram",1,1448),
]

EVENEMENTS_ISLAMIQUES = [
    ("2023-03-16","2023-03-22","Pré-Ramadan","fort"),
    ("2023-03-23","2023-04-20","Ramadan","moyen"),
    ("2023-04-14","2023-04-23","Aïd el-Fitr","très fort"),
    ("2023-06-21","2023-06-30","Aïd el-Adha","très fort"),
    ("2023-07-19","2023-07-19","Nouvel An Islamique","faible"),
    ("2023-09-27","2023-09-27","Mawlid","faible"),
    ("2024-03-04","2024-03-10","Pré-Ramadan","fort"),
    ("2024-03-11","2024-04-08","Ramadan","moyen"),
    ("2024-04-03","2024-04-12","Aïd el-Fitr","très fort"),
    ("2024-06-10","2024-06-19","Aïd el-Adha","très fort"),
    ("2024-07-07","2024-07-07","Nouvel An Islamique","faible"),
    ("2024-09-15","2024-09-15","Mawlid","faible"),
    ("2025-02-22","2025-02-28","Pré-Ramadan","fort"),
    ("2025-03-01","2025-03-29","Ramadan","moyen"),
    ("2025-03-23","2025-04-01","Aïd el-Fitr","très fort"),
    ("2025-05-31","2025-06-09","Aïd el-Adha","très fort"),
    ("2025-06-26","2025-06-26","Nouvel An Islamique","faible"),
    ("2025-09-04","2025-09-04","Mawlid","faible"),
    ("2026-02-10","2026-02-16","Pré-Ramadan","fort"),
    ("2026-02-17","2026-03-17","Ramadan","moyen"),
    ("2026-03-11","2026-03-20","Aïd el-Fitr","très fort"),
    ("2026-05-19","2026-05-28","Aïd el-Adha","très fort"),
    ("2026-06-16","2026-06-16","Nouvel An Islamique","faible"),
]

# Jours fériés tunisiens fixes
JOURS_FERIES = [
    "01-01",  # Nouvel an
    "03-20",  # Fête de l'Indépendance
    "04-09",  # Journée des Martyrs
    "05-01",  # Fête du Travail
    "07-25",  # Fête de la République
    "08-13",  # Fête de la Femme
    "10-15",  # Fête de l'Évacuation
]

def get_mois_islamique(date):
    for debut, fin, mois, num, annee in CALENDRIER_ISLAMIQUE_MOIS:
        if pd.Timestamp(debut) <= date <= pd.Timestamp(fin):
            return mois
    return "Inconnu"

def get_num_mois_islamique(date):
    for debut, fin, mois, num, annee in CALENDRIER_ISLAMIQUE_MOIS:
        if pd.Timestamp(debut) <= date <= pd.Timestamp(fin):
            return num
    return 0

def get_annee_hijri(date):
    for debut, fin, mois, num, annee in CALENDRIER_ISLAMIQUE_MOIS:
        if pd.Timestamp(debut) <= date <= pd.Timestamp(fin):
            return annee
    return 0

def get_evenement(date):
    for debut, fin, evenement, impact in EVENEMENTS_ISLAMIQUES:
        if pd.Timestamp(debut) <= date <= pd.Timestamp(fin):
            return evenement
    return "Période normale"

def get_impact(date):
    for debut, fin, evenement, impact in EVENEMENTS_ISLAMIQUES:
        if pd.Timestamp(debut) <= date <= pd.Timestamp(fin):
            return impact
    return "normal"

def get_saison(mois):
    if mois in (3, 4, 5):   return "Printemps"
    elif mois in (6, 7, 8): return "Été"
    elif mois in (9,10,11): return "Automne"
    else:                    return "Hiver"

def get_tranche_horaire(heure_raw):
    try:
        h = int(str(heure_raw).replace(".0","").zfill(4)[:2])
        if 6  <= h < 10: return "Matin (6h-10h)"
        elif 10 <= h < 13: return "Milieu matin (10h-13h)"
        elif 13 <= h < 15: return "Après-midi (13h-15h)"
        elif 15 <= h < 18: return "Fin après-midi (15h-18h)"
        elif 18 <= h < 21: return "Soir (18h-21h)"
        else:               return "Hors horaires"
    except:
        return "Inconnu"

def est_ferie(date):
    return date.strftime("%m-%d") in JOURS_FERIES

JOURS_FR = ["Lundi","Mardi","Mercredi","Jeudi","Vendredi","Samedi","Dimanche"]

# Générer toutes les dates uniques
dates_uniques = pd.date_range(
    start=df_all["Date"].min(),
    end=df_all["Date"].max(),
    freq="D"
)

print(f"Génération de {len(dates_uniques)} dates...")

dim_temps_rows = []
for i, d in enumerate(dates_uniques):
    dim_temps_rows.append({
        "id":                  i + 1,
        "date_complete":       d.date(),
        "jour":                d.day,
        "mois":                d.month,
        "trimestre":           (d.month - 1) // 3 + 1,
        "annee":               d.year,
        "semaine":             d.isocalendar()[1],
        "num_semaine_mois":    (d.day - 1) // 7 + 1,
        "jour_semaine":        JOURS_FR[d.weekday()],
        "saison":              get_saison(d.month),
        "est_weekend":         d.weekday() >= 5,
        "est_ferie":           est_ferie(d),
        "mois_islamique":      get_mois_islamique(d),
        "num_mois_islamique":  get_num_mois_islamique(d),
        "annee_hijri":         get_annee_hijri(d),
        "evenement_islamique": get_evenement(d),
        "impact_islamique":    get_impact(d),
    })

dim_temps = pd.DataFrame(dim_temps_rows)

print(f"✅ dim_temps : {len(dim_temps)} lignes")
print(f"\nDistribution événements islamiques :")
print(dim_temps["evenement_islamique"].value_counts())
print(f"\nDistribution saisons :")
print(dim_temps["saison"].value_counts())

Génération de 1412 dates...
✅ dim_temps : 1412 lignes

Distribution événements islamiques :
evenement_islamique
Période normale        1208
Ramadan                 116
Aïd el-Adha              40
Pré-Ramadan              28
Aïd el-Fitr              13
Nouvel An Islamique       4
Mawlid                    3
Name: count, dtype: int64

Distribution saisons :
saison
Printemps    368
Été          368
Automne      346
Hiver        330
Name: count, dtype: int64


In [12]:
# ── Cellule 3 : Construire dim_bureau ────────────────────────────────────────

# Mapping gouvernorat par code postal tunisien
# Les codes postaux tunisiens commencent par 2 chiffres qui indiquent le gouvernorat
GOUVERNORAT_PAR_CP = {
    "10": "Tunis", "11": "Tunis", "12": "Tunis", "13": "Tunis",
    "14": "Tunis", "15": "Tunis", "16": "Tunis", "17": "Tunis",
    "18": "Tunis", "19": "Tunis",
    "20": "Ariana", "21": "Ariana", "22": "Ariana",
    "23": "Ben Arous", "24": "Ben Arous",
    "25": "Manouba", "26": "Manouba",
    "30": "Bizerte", "31": "Bizerte", "32": "Bizerte",
    "40": "Nabeul", "41": "Nabeul", "42": "Nabeul",
    "50": "Zaghouan", "51": "Zaghouan",
    "60": "Béja", "61": "Béja",
    "70": "Jendouba", "71": "Jendouba",
    "80": "Kef", "81": "Kef",
    "90": "Siliana", "91": "Siliana",
    "31": "Sousse", "40": "Sousse",
    "40": "Sousse", "41": "Sousse", "42": "Sousse", "43": "Sousse",
    "50": "Monastir", "51": "Monastir", "52": "Monastir",
    "53": "Mahdia", "54": "Mahdia",
    "30": "Sfax", "31": "Sfax", "32": "Sfax", "33": "Sfax", "34": "Sfax",
    "12": "Kairouan", "13": "Kairouan",
    "90": "Kasserine", "91": "Kasserine",
    "93": "Sidi Bouzid", "94": "Sidi Bouzid",
    "60": "Gabès", "61": "Gabès", "62": "Gabès",
    "70": "Médenine", "71": "Médenine",
    "80": "Tataouine", "81": "Tataouine",
    "21": "Gafsa", "22": "Gafsa",
    "23": "Tozeur", "24": "Tozeur",
    "71": "Kébili", "72": "Kébili",
}

# Mapping gouvernorat pour les agences alphabétiques
GOUVERNORAT_AGENCES = {
    99001: "Inconnu",   # 2x34
    99002: "Inconnu",   # AHAM
    99003: "Inconnu",   # AHCH
    99004: "Inconnu",   # AKAS
    99005: "Inconnu",   # AKEF
    99006: "Béja",      # BEJA
    99007: "Bizerte",   # BNZA
    99008: "Gabès",     # GABA
    99009: "Gabès",     # GAFA
    99010: "Jendouba",  # JENA
    99011: "Jendouba",  # JERA
    99012: "Kairouan",  # KAIA
    99013: "Kairouan",  # KEBA
    99014: "Mahdia",    # MAHA
    99015: "Mahdia",    # MEDA
    99016: "Monastir",  # MONA
    99017: "Nabeul",    # NBLA
    99018: "Sfax",      # SFXB
    99019: "Sfax",      # SFXC
    99020: "Sidi Bouzid", # SIDA
    99021: "Sidi Bouzid", # SILA
    99022: "Siliana",   # SSEA
    99023: "Tataouine", # TATA
    99024: "Tabarka",   # TBKA
    99025: "Tozeur",    # TOZA
    99026: "Tunis",     # TUND
    99027: "Tunis",     # TUNE
    99028: "Tunis",     # TUNH
    99029: "Tunis",     # TUNI
    99030: "Tunis",     # TUNK
    99031: "Tunis",     # TUNM
    99032: "Tunis",     # TUNN
    99033: "Tunis",     # TUNO
    99034: "Tunis",     # TUNP
    99035: "Tunis",     # TUNR
    99036: "Tunis",     # TUNU
    99037: "Zaghouan",  # ZARA
}

def get_gouvernorat(id_bureau, cp, ville):
    """Déduire le gouvernorat depuis le code postal ou la ville."""
    id_bureau = int(id_bureau) if pd.notna(id_bureau) else 0

    # Agences alphabétiques → mapping direct
    if id_bureau >= 99001:
        return GOUVERNORAT_AGENCES.get(id_bureau, "Inconnu")

    # Bureaux numériques → via code postal
    cp_str = str(cp).strip().replace(".0", "")
    if len(cp_str) >= 4 and cp_str != "Inconnu":
        prefix2 = cp_str[:2]
        if prefix2 in GOUVERNORAT_PAR_CP:
            return GOUVERNORAT_PAR_CP[prefix2]

    # Fallback → via ville (mots clés)
    ville_upper = str(ville).upper()
    mots_gouvernorat = {
        "TUNIS": "Tunis", "ARIANA": "Ariana", "SFAX": "Sfax",
        "SOUSSE": "Sousse", "BIZERTE": "Bizerte", "NABEUL": "Nabeul",
        "MONASTIR": "Monastir", "MAHDIA": "Mahdia", "KAIROUAN": "Kairouan",
        "GABÈS": "Gabès", "GABES": "Gabès", "GAFSA": "Gafsa",
        "BÉJA": "Béja", "BEJA": "Béja", "JENDOUBA": "Jendouba",
        "KASSERINE": "Kasserine", "SIDI BOUZID": "Sidi Bouzid",
        "TOZEUR": "Tozeur", "KÉBILI": "Kébili", "KEBILI": "Kébili",
        "MÉDENINE": "Médenine", "MEDENINE": "Médenine",
        "TATAOUINE": "Tataouine", "ZAGHOUAN": "Zaghouan",
        "SILIANA": "Siliana", "KEF": "Kef", "MANOUBA": "Manouba",
        "BEN AROUS": "Ben Arous", "TABARKA": "Tabarka",
        "DEGACHE": "Tozeur", "KSAR HLEL": "Monastir",
        "HAMMAM SOUSSE": "Sousse", "MSAKEN": "Sousse",
    }
    for mot, gouv in mots_gouvernorat.items():
        if mot in ville_upper:
            return gouv

    return "Inconnu"

# Extraire les bureaux uniques
bureaux_cols = ["IdBureau", "cp", "cite", "ville", "est_agence"]
df_bureaux = df_all[bureaux_cols].drop_duplicates(subset=["IdBureau"])
df_bureaux = df_bureaux.dropna(subset=["IdBureau"])

# Ajouter gouvernorat
df_bureaux["gouvernorat"] = df_bureaux.apply(
    lambda r: get_gouvernorat(r["IdBureau"], r["cp"], r["ville"]),
    axis=1
)

# Construire dim_bureau
dim_bureau = df_bureaux.reset_index(drop=True)
dim_bureau.insert(0, "id", range(1, len(dim_bureau) + 1))
dim_bureau = dim_bureau.rename(columns={
    "IdBureau": "id_bureau_src",
    "cp":       "code_postal",
    "cite":     "cite",
    "ville":    "ville",
})
dim_bureau["type_bureau"] = dim_bureau["est_agence"].apply(
    lambda x: "Agence" if x else "Bureau"
)
dim_bureau = dim_bureau[["id", "id_bureau_src", "code_postal",
                          "cite", "ville", "gouvernorat", "type_bureau"]]

print(f"✅ dim_bureau : {len(dim_bureau)} bureaux uniques")
print(f"\nDistribution gouvernorats :")
print(dim_bureau["gouvernorat"].value_counts().head(15))
print(f"\nInconnus : {(dim_bureau['gouvernorat'] == 'Inconnu').sum()}")
print(f"\nBureaux vs Agences :")
print(dim_bureau["type_bureau"].value_counts())

✅ dim_bureau : 678 bureaux uniques

Distribution gouvernorats :
gouvernorat
Sfax           93
Sousse         89
Tunis          79
Ariana         79
Monastir       70
Tataouine      60
Gabès          49
Kasserine      37
Gafsa          34
Médenine       31
Kairouan       23
Kébili         13
Inconnu         8
Jendouba        2
Sidi Bouzid     2
Name: count, dtype: int64

Inconnus : 8

Bureaux vs Agences :
type_bureau
Bureau    641
Agence     37
Name: count, dtype: int64


In [13]:
# ── Cellule 4 : Construire dim_service ───────────────────────────────────────

dim_service = pd.DataFrame([
    {
        "id": 1,
        "code_service": "CP",
        "type_service": "Normal",
        "label": "Colis Postal",
        "portee": "National",
        "description": "Envoi standard de colis",
        "tarif_base": None,
        "delai_standard": 5,
    },
    {
        "id": 2,
        "code_service": "UP",
        "type_service": "Normal",
        "label": "Paquet Universel",
        "portee": "National",
        "description": "Paquet léger inférieur à 2kg",
        "tarif_base": None,
        "delai_standard": 5,
    },
    {
        "id": 3,
        "code_service": "RR",
        "type_service": "Normal",
        "label": "Recommandé",
        "portee": "National",
        "description": "Envoi recommandé avec accusé",
        "tarif_base": None,
        "delai_standard": 5,
    },
    {
        "id": 4,
        "code_service": "NOR",
        "type_service": "Normal",
        "label": "Normal",
        "portee": "National",
        "description": "Colis normal",
        "tarif_base": None,
        "delai_standard": 5,
    },
    {
        "id": 5,
        "code_service": "EMS-N",
        "type_service": "Express normal",
        "label": "Express National",
        "portee": "National",
        "description": "Express Mail Service National",
        "tarif_base": None,
        "delai_standard": 2,
    },
    {
        "id": 6,
        "code_service": "EMS-I",
        "type_service": "Express personnalisé",
        "label": "Express International",
        "portee": "International",
        "description": "Express Mail Service International",
        "tarif_base": None,
        "delai_standard": 3,
    },
    {
        "id": 7,
        "code_service": "RPP-I",
        "type_service": "Express personnalisé",
        "label": "Remise Contre Preuve International",
        "portee": "International",
        "description": "Livré par DHL Express",
        "tarif_base": None,
        "delai_standard": 4,
    },
])

print(f" dim_service : {len(dim_service)} services")
print(dim_service[["id", "code_service", "label", "portee"]])

 dim_service : 7 services
   id code_service                               label         portee
0   1           CP                        Colis Postal       National
1   2           UP                    Paquet Universel       National
2   3           RR                          Recommandé       National
3   4          NOR                              Normal       National
4   5        EMS-N                    Express National       National
5   6        EMS-I               Express International  International
6   7        RPP-I  Remise Contre Preuve International  International


In [14]:
# ── Cellule 5 : Construire dim_destination ───────────────────────────────────

# Extraire destinations uniques
dest_cols = ["CodeISOPays", "paysDest", "villeDest", "citeDest", "cpDest", "portee"]
df_dest = df_all[dest_cols].copy()

# Normaliser
df_dest["CodeISOPays"] = df_dest["CodeISOPays"].astype(str).str.strip().str.upper()
df_dest["paysDest"]    = df_dest["paysDest"].astype(str).str.strip().str.upper()
df_dest["villeDest"]   = df_dest["villeDest"].astype(str).str.strip().str.upper()
df_dest["citeDest"]    = df_dest["citeDest"].astype(str).str.strip().str.upper()
df_dest["cpDest"]      = df_dest["cpDest"].astype(str).str.strip()

# Dédupliquer sur les colonnes clés
df_dest = df_dest.drop_duplicates(
    subset=["CodeISOPays", "paysDest", "villeDest", "portee"]
)
df_dest = df_dest.reset_index(drop=True)
df_dest.insert(0, "id", range(1, len(df_dest) + 1))

# Renommer
dim_destination = df_dest.rename(columns={
    "CodeISOPays": "code_iso_pays",
    "paysDest":    "pays_dest",
    "villeDest":   "ville_dest",
    "citeDest":    "cite_dest",
    "cpDest":      "code_postal_dest",
})

print(f" dim_destination : {len(dim_destination)} destinations uniques")
print(f"\nTop 10 pays :")
print(dim_destination["pays_dest"].value_counts().head(10))
print(f"\nRépartition portée :")
print(dim_destination["portee"].value_counts())

 dim_destination : 27920 destinations uniques

Top 10 pays :
pays_dest
TN    12369
FR     3121
IT     1497
US     1419
DE     1258
GB      775
SA      739
CA      604
AE      545
OM      367
Name: count, dtype: int64

Répartition portée :
portee
National         20254
International     7666
Name: count, dtype: int64


In [15]:
# ── Ajouter heure dans fait_colis (colis normaux uniquement) ─────────────────

def formater_heure(heure_raw):
    """Convertit 1520 → 15:20 PM"""
    try:
        h_str = str(heure_raw).replace(".0", "").strip().zfill(4)
        heures = int(h_str[:2])
        minutes = int(h_str[2:])
        if heures < 12:
            return f"{heures:02d}:{minutes:02d} AM"
        elif heures == 12:
            return f"12:{minutes:02d} PM"
        else:
            return f"{heures-12:02d}:{minutes:02d} PM"
    except:
        return None

def get_tranche(heure_raw):
    try:
        h = int(str(heure_raw).replace(".0","").strip().zfill(4)[:2])
        if 6  <= h < 10:  return "Matin (6h-10h)"
        elif 10 <= h < 13: return "Milieu matin (10h-13h)"
        elif 13 <= h < 15: return "Après-midi (13h-15h)"
        elif 15 <= h < 18: return "Fin après-midi (15h-18h)"
        elif 18 <= h < 21: return "Soir (18h-21h)"
        else:               return "Hors horaires"
    except:
        return "Inconnu"

# Ajouter dans df_all
df_all["heure_formatee"] = df_all["Heure"].apply(
    lambda x: formater_heure(x) if pd.notna(x) else None
)
df_all["tranche_horaire"] = df_all["Heure"].apply(
    lambda x: get_tranche(x) if pd.notna(x) else "Non applicable"
)

print(" Heure formatée")
print("\nExemples :")
print(df_all[df_all["type_source"] == "normal"][
    ["Heure", "heure_formatee", "tranche_horaire"]
].head(10))

print("\nDistribution tranches horaires (normaux) :")
print(df_all[df_all["type_source"] == "normal"]["tranche_horaire"].value_counts())

print("\nExpress → heure :")
print(df_all[df_all["type_source"] == "express"]["tranche_horaire"].value_counts())

 Heure formatée

Exemples :
  Heure heure_formatee           tranche_horaire
0  1520       03:20 PM  Fin après-midi (15h-18h)
1  1031       10:31 AM    Milieu matin (10h-13h)
2  0819       08:19 AM            Matin (6h-10h)
3  0824       08:24 AM            Matin (6h-10h)
4  0841       08:41 AM            Matin (6h-10h)
5  0950       09:50 AM            Matin (6h-10h)
6  1104       11:04 AM    Milieu matin (10h-13h)
7  1014       10:14 AM    Milieu matin (10h-13h)
8  1134       11:34 AM    Milieu matin (10h-13h)
9  0912       09:12 AM            Matin (6h-10h)

Distribution tranches horaires (normaux) :
tranche_horaire
Milieu matin (10h-13h)      165506
Matin (6h-10h)              110343
Fin après-midi (15h-18h)     76051
Après-midi (13h-15h)         29014
Soir (18h-21h)                 913
Hors horaires                  380
Name: count, dtype: int64

Express → heure :
tranche_horaire
Non applicable    865453
Name: count, dtype: int64


In [16]:
# ── Cellule 6 : Construire fait_colis ────────────────────────────────────────

# Créer les dictionnaires de lookup pour les IDs
print("Construction des tables de lookup...")

# Lookup dim_temps : date → id
temps_lookup = dict(zip(
    dim_temps["date_complete"].astype(str),
    dim_temps["id"]
))

# Lookup dim_bureau : id_bureau_src → id
bureau_lookup = dict(zip(
    dim_bureau["id_bureau_src"].astype(str),
    dim_bureau["id"]
))

# Lookup dim_service : code_service → id
service_lookup = dict(zip(
    dim_service["code_service"],
    dim_service["id"]
))

# Lookup dim_destination : (code_iso_pays, pays_dest, ville_dest, portee) → id
dest_lookup = {}
for _, row in dim_destination.iterrows():
    key = (
        str(row["code_iso_pays"]).strip().upper(),
        str(row["pays_dest"]).strip().upper(),
        str(row["ville_dest"]).strip().upper(),
        str(row["portee"]).strip(),
    )
    dest_lookup[key] = row["id"]

print(" Lookups construits")

# Construire fait_colis
print("Construction fait_colis...")

fait_colis = df_all[[
    "NumEnvoi", "IdBordereau", "Poids", "Montant",
    "RefPaiement", "type_source", "CodeService",
    "Date", "IdBureau", "CodeISOPays",
    "paysDest", "villeDest", "portee",
]].copy()

# Mapper les IDs
fait_colis["temps_id"] = fait_colis["Date"].dt.date.astype(str).map(temps_lookup)

fait_colis["bureau_id"] = fait_colis["IdBureau"].astype(str).map(bureau_lookup)

fait_colis["service_id"] = fait_colis["CodeService"].map(service_lookup)

fait_colis["destination_id"] = fait_colis.apply(
    lambda r: dest_lookup.get((
        str(r["CodeISOPays"]).strip().upper(),
        str(r["paysDest"]).strip().upper(),
        str(r["villeDest"]).strip().upper(),
        str(r["portee"]).strip(),
    )), axis=1
)

# Renommer et sélectionner colonnes finales
fait_colis = fait_colis.rename(columns={
    "NumEnvoi":    "num_colis",
    "IdBordereau": "id_bordereau",
    "Poids":       "poids",
    "Montant":     "montant",
    "RefPaiement": "ref_paiement",
})

fait_colis = fait_colis[[
    "num_colis", "id_bordereau", "poids", "montant",
    "ref_paiement", "type_source",
    "temps_id", "bureau_id", "service_id", "destination_id",
]]
# Dans la construction de fait_colis — après la ligne type_source
fait_colis["heure_formatee"]  = df_all["heure_formatee"].values
fait_colis["tranche_horaire"] = df_all["tranche_horaire"].values
# Ajouter colonnes ML (nulles pour l'instant)
fait_colis["nature"]       = None
fait_colis["est_anomalie"] = None

# Ajouter ID
fait_colis.insert(0, "id", range(1, len(fait_colis) + 1))

# Vérifier les nulls sur les clés étrangères
print(f"\n fait_colis : {len(fait_colis):,} lignes")
print(f"\nClés étrangères nulles :")
print(f"  temps_id       : {fait_colis['temps_id'].isnull().sum():,}")
print(f"  bureau_id      : {fait_colis['bureau_id'].isnull().sum():,}")
print(f"  service_id     : {fait_colis['service_id'].isnull().sum():,}")
print(f"  destination_id : {fait_colis['destination_id'].isnull().sum():,}")

Construction des tables de lookup...
 Lookups construits
Construction fait_colis...

 fait_colis : 1,247,660 lignes

Clés étrangères nulles :
  temps_id       : 0
  bureau_id      : 0
  service_id     : 0
  destination_id : 0


In [17]:
# ── Cellule 7 : Sauvegarder les dimensions ───────────────────────────────────

tables = {
    "dim_temps":       dim_temps,
    "dim_bureau":      dim_bureau,
    "dim_service":     dim_service,
    "dim_destination": dim_destination,
    "fait_colis":      fait_colis,
}

for nom, df in tables.items():
    chemin = PATH_OUT + f"{nom}.csv"
    df.to_csv(chemin, index=False, encoding="utf-8")
    print(f" {nom}.csv — {len(df):,} lignes")

print(f"\n Transformation terminée — 5 tables prêtes pour l'ETL")

 dim_temps.csv — 1,412 lignes
 dim_bureau.csv — 678 lignes
 dim_service.csv — 7 lignes


 dim_destination.csv — 27,920 lignes
 fait_colis.csv — 1,247,660 lignes

 Transformation terminée — 5 tables prêtes pour l'ETL
